# Gain-scheduled DeePC on Reacher — composite regional data
An independent Python/CVXPY port checked against **Zieglmeier et al., arXiv:2512.02797v2**, and their MATLAB implementation. This is a new Reacher adaptation, **not an authors' Reacher experiment or a validated/tuned baseline**.

All execution switches are off initially. Start with the data audit and smoke test. Do not interpret a short smoke rollout as a success-rate result. This notebook does not run Atlas, Select-DPC, contextual DPC, or modify their saved results.


## Related papers and implementation choice
1. **Margarita A. Guerrero, Braghadeesh Lakshminarayanan, Cristian R. Rojas (2025)**, [Gain-Scheduled Data-Enabled Predictive Control: A DeePC Approach for Nonlinear Systems](https://arxiv.org/abs/2509.26334). Local Hankel banks, measured scheduling signals, hysteresis and input/parameter blending; ship-steering example.
2. **Sebastian Zieglmeier, Mathias Hudoba de Badyn, Narada D. Warakagoda, Thomas R. Krogstad, Paal Engelstad (2025; v2 February 2026)**, [Gain-Scheduling Data-Enabled Predictive Control for Nonlinear Systems with Linearized Operating Regions](https://arxiv.org/abs/2512.02797v2). Regional mosaic-Hankel matrices and composite-region switching; unbalanced-disc example. **Implemented here.**
3. Related application: Zieglmeier, Verhoek, Eising and Hudoba de Badyn, [Model-Free Control for Residential Heating: Deployment and Simulation of Nonlinear Data-Enabled Predictive Control](https://arxiv.org/abs/2609.21500) (2026). This is not the implementation source.

Primary code: [SebsDevLab/GS_DeePC](https://github.com/SebsDevLab/GS_DeePC), inspected commit `8738264c101c6e2942735ada3dc6bfb24ee5a101`.
Checked files: `GS_DeePC/controllers/DeePC_fast_GS.m`, `GS_DeePC/GS_DeePC_overlapping.m`, and `GS_DeePC/functions/make_hankel_MIMO.m`.
The notebook needs no MATLAB, MOSEK, internet connection, or upstream checkout to run.

**Paper/code distinction:** the public controller enforces the terminal equality below, although printed Eq. (3) omits it. We follow the code and retain it. The switching port handles jumps across multiple regions and the final domain endpoint explicitly; these boundary-safety changes do not add another selector or fallback data bank.


## Controller and data construction
At each step, use one active composite bank and solve
\[
\min_{g,u,y,\sigma}\sum_{j=0}^{N-1}
Q_{xy}\|y_j-r_j\|_2^2+R\|u_j\|_2^2
+\lambda_g\|g\|_2^2+\lambda_{\mathrm{ini}}\|\sigma\|_2^2
\]
subject to
\[
U_pg=u_{\mathrm{ini}},\quad Y_pg=y_{\mathrm{ini}}+\sigma,\quad
U_fg=u,\quad Y_fg=y,\quad y_{N-1}=r_{N-1},
\]
and physical input/output box bounds. Apply the first input and repeat.

Raw windows are kept only if **every sample stays in the same elementary region**, without crossing episode boundaries. Following upstream, keep the first 200 chronological columns per region. A composite concatenates two neighboring banks (400 columns). Keep the current composite until the measured scheduler leaves its interval.

No SVD compression, affine sum constraint, projection penalty, input increments, input augmentation, smoothing cost, refinement loop, Atlas chart selector, graph, or terminal-posture construction is added. Singular values in the data audit are diagnostics only.


## Reacher-specific choices and limitations
| Item | Setting |
|---|---|
| Offline data | Same frozen baseline `data/reacher/select_iid_offline_hankel.npy`; recover all original episodes with overlap checks |
| Controlled outputs | Fingertip x/y, in metres; all 8 baseline outputs also recorded |
| Scheduling variable | Measured wrapped second joint angle: atan2(sin(q2), cos(q2)) |
| Regions | 8 equal elementary intervals on [-π, π], giving 7 composites |
| Representation | 200 raw columns per elementary region; 400 active raw coefficients |
| Horizons | T_ini=2, T_f=N=10, matching contextual DPC; explicit change from the paper's N=5 |
| Weights | Paper disc values Q=100, R=0.05, λ_ini=1e6, λ_g=1e3; **not Reacher-tuned** |
| Constraints | Reacher inputs ±1; predicted x/y within ±0.22 m; terminal x/y equals final target |
| Reference | Fixed environment target, no intermediate waypoint or joint posture |
| Initialization | Reacher-v5 reset; 12 zero-action warmup steps, using real measured histories |
| Evaluation | Seeds 211–310 inclusive; 320 controlled steps; serial, single numerical thread |
| Solver | Clarabel instead of authors' MOSEK; parametric QP built once |
| Rejected solve | Explicit zero-input fallback, as in upstream; status and fallback count saved |

The scheduler is a **design hypothesis**, not a proven LPV embedding: q2 reflects configuration-dependent coupling but does not capture every dependence on q1, velocities, or Cartesian kinematics. Wrapped-angle discontinuities are not connected across -π/π. These are linear ordered bins, not a circular atlas. Eight regions is a disclosed starting choice, not a selected optimum.

Raw physical coordinates are used without centering or standardization. Paper weights therefore have different physical meaning on Reacher. Full row rank of the input bank at L is checked, but it is not proof of PE of order L+n or of local LTI behavior. Using the existing data also differs from the authors' targeted disc excitation.

Do not compare method-specific weighted costs directly across algorithms with different objectives. The saved unweighted squared XY error and input effort support a common external metric.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, signal
from dataclasses import asdict
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/atlas_deepc').is_dir() and (p / 'data/reacher').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the atlas_deepc_reacher project.')
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[key] = '1'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from atlas_deepc.benchmarks import gain_scheduled_deepc_reacher as gs
print('Project:', ROOT)
print('Implementation:', gs.__file__)


## Weight-only tuning result — no working candidate found
Expanded study, October 16, 2026: 36 distinct weight sets were tested in 90 development rollouts, with T_ini=2 and T_f=10 fixed. Ten initial seed-211 pilots used 200 steps; subsequent checks used 320 steps. The expansion added 22 settings on seeds 211/222, then completed a matched eight-seed panel for the original weights and two lowest-error screened candidates. **None of the 90 rollouts met strict success. This is not a 100-seed success-rate estimate.**

All saved offline banks were verified identical across candidates. Only Q_xy, R, lambda_g and lambda_ini changed; the raw representation, 8-region scheduler, terminal equality, and solver were unchanged. There were no solver-fallback steps in these trials.

On the matched eight-seed/320-step panel (211, 219, 222, 237, 258, 281, 294, 305), Q=1000, R=5, lambda_g=10, lambda_ini=1e6 had the lowest mean final-100 error (84.36 mm versus 166.85 mm for the paper weights), but still scored 0/8. The Q=100/R=1/lambda_g=10 candidate scored 0/8 with 119.21 mm mean error. **No setting is promoted as a successful tuned baseline.** The configuration below retains the original weights as an explicitly untuned reference; execution remains disabled.

The separate scheduler study below investigates the q2-only local-data hypothesis. This is an investigation of this Reacher adaptation, not a failure claim about the paper. The cell below only reads saved evidence.


In [ ]:
tuning_path = ROOT / 'experiments/reacher/gs_deepc_control_tuning_tf10_v1/tuning_report.json'
if tuning_path.exists():
    tuning = json.loads(tuning_path.read_text())
    print('Weight sets:', tuning['unique_weight_sets'], '| Completed rollouts:', tuning['completed_rollouts'],
          '| Successful rollouts:', tuning['successful_rollouts'])
    print('Matched eight-seed panel, 320 steps (development checks, not held-out evaluation):')
    display(pd.DataFrame(tuning.get('matched_eight_seed_320', [])))
    print('Top ten settings on the matched two-seed screen, 320 steps:')
    display(pd.DataFrame(tuning.get('matched_two_seed_320', [])).head(10))
    print('All pilot configurations (different seed panels/lengths; do not rank together):')
    display(pd.DataFrame(tuning['candidates']))
else:
    print('No saved tuning report:', tuning_path)


In [ ]:
# Read-only scheduler study: no simulations are launched.
scheduler_path = ROOT / 'experiments/reacher/gs_deepc_scheduler_pilot_tf10_v1/scheduler_report.json'
if scheduler_path.exists():
    scheduler_report = json.loads(scheduler_path.read_text())
    print('Scheduler-only Reacher adaptations; matched eight development seeds, 320 steps.')
    print('Fixed Q=1000, R=5, lambda_g=10, lambda_ini=1e6; unchanged raw-column QP.')
    display(pd.DataFrame([{k: v for k, v in row.items() if k != 'results'}
                          for row in scheduler_report['schedulers']]))
    audit = scheduler_report['fixed_input_prediction_audit']
    print('Separate fixed-input prediction diagnostic:', audit['windows'], 'windows from', audit['episodes'], 'unused offline episodes.')
    display(pd.DataFrame(audit['schedulers']).T)
else:
    print('No saved scheduler report:', scheduler_path)


### Scheduler-only pilot: limited improvement, not a working baseline
The same eight development seeds were checked with measured q2, q1, and wrapped q1+q2 (second-link orientation) as scalar scheduling coordinates. All used T_ini=2, T_f=10, 8 elementary regions, 200 raw columns per region, and the same fixed weights shown above. The q2 results were reused; 16 new rollouts tested the alternatives. Only the scheduling coordinate and consequently the retained regional windows changed. The QP, terminal equality, raw physical u/xy, and switching logic were unchanged. These are explicitly Reacher-specific design choices, not authors' reported experiments.

Strict successes: q2 **0/8**, q1 **1/8**, q1+q2 **1/8**. Mean final-100 errors: **84.36, 129.92, 54.70 mm**, respectively. No solver fallbacks occurred. q1 succeeded on seed 222; q1+q2 succeeded on seed 211. No setting is promoted; the original configuration and execution switches below remain unchanged.

A separate prediction audit uses 304 non-overlapping windows from offline episodes 181--199 (zero-based), excluded from every retained bank. It fits g using the same quadratic coefficient/past-output penalties, exact past AND known future inputs, then predicts future xy. It has no target, terminal equality, future-output penalty, or output box: this is a diagnostic predictor, NOT a changed closed-loop controller. Initial composite selection uses the measured scheduler at the last past sample without switching history. Mean ten-step trajectory RMSE remains about 89--93 mm across schedulers. Staying-in-composite statistics use different subsets and should not be treated as a paired comparison. Closed-loop prediction statistics likewise reflect different visited trajectories.

Conclusion: scalar scheduling affects performance, but these tests do not isolate a unique failure cause or establish that the published method fails. Possible remaining issues include short XY-only history, regional coverage, trajectory departures from the active region, and model validity. The saved report is in `experiments/reacher/gs_deepc_scheduler_pilot_tf10_v1/scheduler_report.json`.


In [ ]:
# Read saved history-length diagnostics only; this does not run simulations.
history_path = ROOT / 'experiments/reacher/gs_deepc_history_pilot_tf10_v1/history_report.json'
if history_path.exists():
    history_report = json.loads(history_path.read_text())
    print('Paired seeds 211/222, 320 steps; separate history-length adaptation.')
    display(pd.DataFrame([{k: v for k, v in row.items() if k != 'results'}
                          for row in history_report['two_seed_screen']]))
    audit = history_report['prediction_audit']
    print('Fixed-input audit:', audit['windows'], 'paired windows from',
          len(audit['paired_unused_episodes_zero_based']), 'episodes absent from all retained banks.')
    display(pd.DataFrame(audit['results']))
    print('Settings rejected for insufficient regional data:')
    display(pd.DataFrame(audit['unavailable']))
else:
    print('No saved history report:', history_path)


### Longer XY history did not improve the control screen
A separate diagnostic tested T_ini=2,3,4,6,8 with T_f=10, 8 elementary regions, 200 raw columns per region, and fixed Q=1000, R=5, lambda_g=10, lambda_ini=1e6. Sufficient regional data existed through T_ini=4 for q2 and T_ini=3 for q1+q2. Infeasible data settings were rejected, without changing column count or inserting fallback data.

For prediction, all feasible settings used the same 532 future windows from 38 episodes absent from every retained bank. Future sample times were aligned across history lengths. The known-input regularized diagnostic is the same type as above, not the tracking QP. q2 T_ini=2 to 4 improved mean one-step error from 11.36 to 6.31 mm, but mean ten-step RMSE only from 94.77 to 90.54 mm. q1+q2 T_ini=2 to 3 improved these from 9.87 to 6.73 mm and 107.95 to 104.58 mm. These values use a different paired test set than the preceding scheduler audit; do not compare their absolute values directly.

Four new 320-step rollouts then tested q2/T_ini=4 and q1+q2/T_ini=3 on seeds 211 and 222. Both scored **0/2**, without solver fallbacks. Mean final-100 errors were 96.40 and 27.83 mm, versus 81.75 and 17.54 mm for their respective saved T_ini=2 runs (0/2 and 1/2 successes). The q1+q2/T_ini=3 seed-222 final-100 mean was 10.58 mm, slightly above the 10 mm threshold; its final-30 criterion passed.

This branch was stopped after the two-seed screen; no eight- or 100-seed extension was launched. Longer history changes which within-region raw windows are retained, so this is not a pure observability test. The result does not establish a unique failure cause. The matched T_ini=2 benchmark, original controller, and configuration below remain unchanged.


In [ ]:
# Read-only: region-count and existing-weight tuning, no new controller mechanism.
region_path = ROOT / 'experiments/reacher/gs_deepc_region_tuning_tf10_v1/region_tuning_report.json'
if region_path.exists():
    region_report = json.loads(region_path.read_text())
    print('New completed rollouts:', region_report['new_completed_rollouts'])
    print('Raw banks identical across weight changes:', region_report['raw_banks_identical_across_weight_changes'])
    for panel in ('eight_seed_panel', 'two_seed_screen'):
        print(panel, '(development tuning, not held-out evaluation)')
        display(pd.DataFrame([{k: v for k, v in row.items() if k != 'results'}
                              for row in region_report[panel]]))
else:
    print('No saved region/weight report:', region_path)


### Parameter-only tuning within the published GS-DeePC structure
Rechecked against [the paper, Section 5](https://arxiv.org/html/2512.02797v2) and the public MATLAB files `GS_DeePC_overlapping.m`, `GS_DeePC_overlapping_looped.m`, and `controllers/DeePC_fast_GS.m` at the pinned commit above. The authors explicitly vary region count; their looped experiment sweeps 2--50 elementary regions. Region count and the existing quadratic weights are therefore tuning parameters, not additional controller mechanisms.

This study varies elementary regions (4, 6, 8) and R/lambda_g while retaining Q=1000 and lambda_ini=1e6. It uses the previously tested measurable q2 and wrapped q1+q2 scalar schedulers. T_ini=2, T_f=10, the baseline offline dataset, chronological first-200 raw columns per elementary region, two-bank composites (400 coefficients), switching rule, input/output bounds, terminal equality from the authors' code, and solver all stay unchanged. No SVD, affine sum constraint, graph, waypoint, projection regularizer, new penalty, or new fallback has been added. For each fixed scheduler/region count, saved raw matrices and provenance are checked identical across weight trials.

The two-seed screen uses seeds 211 and 222, 320 steps. Promising settings are checked on six more development seeds (219, 237, 258, 281, 294, 305), giving paired eight-seed panels. All failures remain in the report and the original strict success criterion is unchanged. Weighted native costs are not used to rank different weight settings. These remain Reacher adaptations and a Python/Clarabel port, not a verbatim reproduction of the authors' unbalanced-disc experiment. Original baseline execution settings below are not silently replaced.

Completed round: **12 new parameter settings, 36 new rollouts**, no solver-fallback steps. With eight elementary regions and q1+q2 scheduling, R=0.5 and R=1 each passed both screening seeds but failed all six additional seeds, ending at **2/8**. Their mean final-100 errors were **30.75 mm** and **58.24 mm**, respectively. The previous R=5 setting scored 1/8 and 54.70 mm on the same panel. Other weights were fixed at Q=1000, lambda_g=10, lambda_ini=1e6. The R=0.5 setting is the best candidate of this round, **not a validated working baseline**; no setting was promoted and no 100-seed run was launched.


In [ ]:
# Keep new parameter experiments in NEW output directories: provenance is enforced.
CONFIG = gs.Config(tini=2, horizon=10, regions=8, columns_per_region=200,
                   q_xy=100., r=0.05, lambda_ini=1e6, lambda_g=1e3)
SEED_START, SEED_STOP, STEPS = 211, 310, 300
RESULT_DIR = ROOT / 'experiments/reacher/gs_deepc_composite_xy_211_310_tini2_tf10_v1'
SMOKE_DIR = ROOT / 'experiments/reacher/gs_deepc_composite_xy_smoke_tini2_tf10_v1'
SMOKE_SEED, SMOKE_STEPS = 211, 20
PREPARE_OFFLINE = False
RUN_SMOKE = False
RUN_PANEL = False
CONFIG.validate()
display(pd.DataFrame([asdict(CONFIG)]))
print('Active raw columns per QP:', 2 * CONFIG.columns_per_region)


In [ ]:
def command(output, start, stop, steps, prepare=False):
    cmd = [sys.executable, '-m', 'atlas_deepc.benchmarks.gain_scheduled_deepc_reacher',
           '--seed-start', str(start), '--seed-stop', str(stop), '--steps', str(steps),
           '--output-dir', str(output)]
    exposed = ('tini', 'horizon', 'regions', 'columns_per_region', 'warmup',
               'q_xy', 'r', 'lambda_ini', 'lambda_g', 'xy_bound')
    for key in exposed:
        cmd += ['--' + key.replace('_', '-'), str(getattr(CONFIG, key))]
    if prepare:
        cmd += ['--prepare-only']
    return cmd

def launch(cmd):
    env = os.environ.copy()
    env['PYTHONPATH'] = str(ROOT / 'src')
    process = subprocess.Popen(cmd, cwd=ROOT, env=env, start_new_session=True)
    try:
        return_code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=15)
        print('Stopped. Completed seeds remain saved; the interrupted seed will restart.')
        raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)


## 1. Offline preparation and audit
Set PREPARE_OFFLINE=True above and run this cell. No plant rollouts are performed. The audit checks column availability and input rank; missing regional data is an error, not silently replaced with a global bank. Preparation stores matrices and original (episode, start) IDs in offline_library.npz. On later runs deterministic preparation is repeated outside the online timer and provenance is checked.


In [ ]:
if PREPARE_OFFLINE:
    launch(command(RESULT_DIR, SEED_START, SEED_STOP, STEPS, prepare=True))
audit_path = RESULT_DIR / 'data_audit.json'
if audit_path.exists():
    audit = json.loads(audit_path.read_text())
    display(pd.DataFrame(audit['regions']))
    display(pd.DataFrame({'region': range(CONFIG.regions),
                          'available_columns': audit['valid_columns_per_region']}))
    print(audit['excitation_caveat'])
else:
    print('Not prepared yet. Enable PREPARE_OFFLINE and rerun this cell.')


## 2. Short smoke test
Enable RUN_SMOKE to check end-to-end execution on one seed. Defaults to 20 steps, **not** a success test. Inspect fallback counts, constraint residuals, predicted versus observed motion, and switching before enabling the panel. No tuning is performed automatically.

Historical implementation check with N=5 (October 16, 2026): 20 steps on seed 211 completed with zero solver fallbacks and one region switch. Error changed from about 49 to 80 mm. These are NOT results for the current N=10 configuration and do not demonstrate successful tracking. Historical evidence is preserved in `experiments/reacher/gs_deepc_implementation_check_final_v1`. The current horizon uses separate output directories to avoid mixing results.


In [ ]:
if RUN_SMOKE:
    launch(command(SMOKE_DIR, SMOKE_SEED, SMOKE_SEED, SMOKE_STEPS))
else:
    print('Smoke test disabled.')
smoke_summary = SMOKE_DIR / 'summary.json'
if smoke_summary.exists():
    display(pd.DataFrame(json.loads(smoke_summary.read_text())['results']))


## 3. Run or resume the development panel
Enable RUN_PANEL only when ready. Completed seeds are reused only under an identical settings/source/data/package manifest. A changed configuration requires a new directory. Interrupting stops the child process; the current incomplete seed is restarted on resume.

Success uses the shared criterion: mean error <10 mm and maximum error <20 mm in **both** the last 30 and last 100 control steps. Outcomes with fallback actions remain included and their fallback counts are shown. A short run has no success-rate estimate.


In [ ]:
if RUN_PANEL:
    launch(command(RESULT_DIR, SEED_START, SEED_STOP, STEPS))
else:
    print('100-seed panel disabled. No baseline algorithms will be rerun.')
progress_path = RESULT_DIR / 'progress.json'
if progress_path.exists():
    print(json.loads(progress_path.read_text()))


## 4. Saved results and modular computation time — no simulations
Rerun these display cells any time. They only read saved results. Error quantiles summarize per-rollout final-100 mean errors, including failed tasks. Rollout RMSE includes every controlled step. Times pool steps 6 onward and include unsuccessful solves.

Total online time includes selection, parameter assignment, the solve call, and acceptance diagnostics. The solve-call timer includes CVXPY canonicalization, solver setup and numerical solution; it is not solver iterations alone. The first solve's compilation is included in the all-step timer. Static problem construction and data preparation are offline. Simulation and file I/O are excluded. Component medians need not sum to total median.


In [ ]:
summary_path = RESULT_DIR / 'summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.exists() else None
if summary is None:
    print('No saved panel results yet:', RESULT_DIR)
else:
    print(f"Saved {summary['completed']}/{summary['requested']} seeds; complete={summary['complete']}")
    if summary['success_rate'] is not None:
        print(f"Strict success: {summary['successes']}/{summary['success_evaluable_rollouts']} "
              f"({100*summary['success_rate']:.1f}%)")
    print('Fallback steps:', summary['fallback_steps'])
    display(pd.DataFrame(summary['results']))
    if summary['error_quantiles_mm']:
        display(pd.DataFrame([summary['error_quantiles_mm']], index=['Final-100 mean error (mm)']))
    timing = summary['modular_timing_steps6_onwards']
    display(pd.DataFrame({key: val for key, val in timing.items() if val is not None}).T)
    print('All-step total timing:', summary['timing_all_steps'])


In [ ]:
if summary is not None and summary['results']:
    results = pd.DataFrame(summary['results'])
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    eligible = results[results['success_evaluable']]
    if len(eligible):
        axes[0].hist(1000*eligible['mean_last_100'], bins=20)
        axes[0].axvline(10, color='red', linestyle='--', label='10 mm mean threshold')
        axes[0].legend(fontsize=8)
    axes[0].set(xlabel='Final-100 mean error [mm]', ylabel='Rollouts')
    axes[1].plot(results['seed'], results['closed_loop_control_cost'], '.')
    axes[1].set(xlabel='Seed', ylabel='Realized tracking + effort cost', ylim=(0, None))
    timing = summary['modular_timing_steps6_onwards']
    names = ['selection_time_s', 'matrix_preparation_time_s', 'solve_time_s']
    if all(timing[name] for name in names):
        axes[2].bar(['Selection', 'Preparation', 'QP call'],
                    [timing[name]['mean_ms'] for name in names])
    axes[2].set(ylabel='Mean computation [ms/step]', ylim=(0, None))
    fig.suptitle(f"GS-DeePC: {summary['completed']}/{summary['requested']} saved seeds (untuned)")
    fig.tight_layout()
    plt.show()


## 5. Inspect one rollout
Choose SMOKE_DIR or RESULT_DIR below. Predictions correspond to the action applied at that step and the measured **post-action** fingertip output. NaN predictions mark rejected solves/zero-input fallback; they are not plotted as accurate predictions.


In [ ]:
VIEW_DIR, VIEW_SEED = SMOKE_DIR, SMOKE_SEED
trace_path = VIEW_DIR / f'seed{VIEW_SEED}_control_results.npz'
if trace_path.exists():
    with np.load(trace_path) as saved:
        trace = {key: saved[key].copy() for key in saved.files}
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    axes[0,0].plot(1000*trace['distance'], label='Target error')
    axes[0,0].plot(1000*trace['prediction_error_xy'], label='One-step prediction error', alpha=.8)
    axes[0,0].set(ylabel='Error [mm]'); axes[0,0].legend()
    axes[0,1].plot(trace['y'][:,6], trace['y'][:,7], label='Fingertip')
    axes[0,1].scatter(*trace['target_xy'], marker='*', s=100, label='Target')
    points = np.vstack([trace['y'][:,6:8], trace['target_xy']])
    center = (points.min(axis=0) + points.max(axis=0)) / 2
    radius = max(float(np.ptp(points, axis=0).max()) * 0.65, 0.02)
    axes[0,1].set(xlabel='x [m]', ylabel='y [m]', aspect='equal',
                  xlim=(center[0]-radius, center[0]+radius), ylim=(center[1]-radius, center[1]+radius))
    axes[0,1].legend(fontsize=8)
    axes[1,0].plot(trace['scheduler_q2'], label='q2 [rad]')
    axes[1,0].step(np.arange(len(trace['distance'])), trace['composite_index'], label='Composite ID', where='post')
    axes[1,0].set(xlabel='Control step'); axes[1,0].legend()
    for field, label in [('control_compute_time_s','Total'), ('selection_time_s','Selection'),
                         ('matrix_preparation_time_s','Preparation'), ('solve_time_s','QP call')]:
        axes[1,1].plot(1000*trace[field], label=label)
    axes[1,1].set(xlabel='Control step', ylabel='Computation [ms, log scale]', yscale='log'); axes[1,1].legend()
    fig.suptitle(f"GS-DeePC seed {VIEW_SEED}: {int(trace['fallback'].sum())} fallback steps")
    fig.tight_layout(); plt.show()
else:
    print('No saved trace:', trace_path)
